In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import sys
import os

sys.path.append('..')
os.chdir('..')  

from src.utils.config_loader import load_config
from src.data_pipeline.preprocess import *
from src.data_pipeline.features import *
config = load_config("configs/data_config.yaml")
print("Config loaded successfully ")

https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz
Config loaded successfully 


In [2]:
# Load ratings data
raw_path = config['paths']['raw_data']

df_ratings = pd.read_parquet(raw_path + "Electronics_ratings.parquet")

print(f"Shape: {df_ratings.shape}")
print(f"\nColumns: {df_ratings.columns.tolist()}")
print(f"\nFirst 5 rows:")
df_ratings.head()

Shape: (1000000, 10)

Columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

First 5 rows:


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3.0,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,"[{'attachment_type': 'IMAGE', 'large_image_url...",B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1658185117948,0,True
1,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1592678549731,0,True
2,5.0,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1523093017534,0,True
3,5.0,Great laptop backpack!,I was searching for a sturdy backpack for scho...,[],B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,1290278495000,18,True
4,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,[],B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1676601581238,0,True


In [3]:
# 1. CLEANING & FORMATTING
prep_config = config['preprocessing']
df_ratings = drop_useless_columns(df_ratings, prep_config['drop_columns'])
df_ratings = remove_missing_values(df_ratings, prep_config['missing_values']['subset'])
df_ratings = remove_duplicates(df_ratings)
df_ratings = convert_timestamp(df_ratings)
df_ratings = convert_to_integer(df_ratings, prep_config['convert_to_integer']['columns'])
df_ratings = handle_outliers(df_ratings, prep_config['outliers']['columns'])

# 2. SPAM & TEXT FILTERING
df_ratings = detect_spam(df_ratings, prep_config['spam_detection']['max_reviews_per_day'], prep_config['spam_detection']['min_time_gap'])
df_ratings = filter_text(df_ratings, column='text',
                         min_words=prep_config['text_filter']['min_words'],
                         max_words=prep_config['text_filter']['max_words'])

# 3. HIGH-SIGNAL FILTERING
df_ratings = filter_high_signal_ratings(df_ratings, min_rating=4.0)

# 4. DEDUPLICATION
df_ratings = deduplicate_user_item(df_ratings, user_col='user_id', item_col='parent_asin')

# 5. DENSITY FILTERING
k_core_config = {'user_id': 5, 'parent_asin': 5}
df_ratings = apply_iterative_k_core(df_ratings, k_core_config)

# 6. POPULARITY REDUCTION
if len(df_ratings) > 10000:
    df_ratings = filter_top_n_items(df_ratings, item_col='parent_asin', top_n=10000)
else:
    print("[INFO] Skipping Top-N filtering because dataset is small.")
# 7. CONFIDENCE WEIGHTING
df_ratings = apply_confidence_weight(
    df_ratings, 
    verified_col='verified_purchase', 
    weight_col='interaction_weight'
)
print(f"[DEBUG] Rows before split: {len(df_ratings)}")
if len(df_ratings) == 0:
    print("[ERROR] The DataFrame is empty! The filtering steps removed everything.")
# 8. ENCODING & SPLITTING
df_ratings, encoders = encode_labels(df_ratings, ['user_id', 'parent_asin'])
train_df, val_df, test_df = time_based_split(
    df_ratings, 
    column="timestamp", 
    val_year=prep_config['split']['val_year'], 
    test_year=prep_config['split']['test_year']
)

[INFO] Dropped columns: ['images']
[INFO] Removed 220 rows with missing values | Remaining: 999780
[INFO] Removed 220 rows with missing values | Remaining: 999780
[INFO] Removed 2495 duplicate rows | Remaining: 997285
[INFO] Converted 'timestamp' to datetime format
[INFO] Converted 'verified_purchase' to integer type
[INFO] Applied log1p transformation to 'helpful_vote'
[INFO] Removed 2495 duplicate rows | Remaining: 997285
[INFO] Converted 'timestamp' to datetime format
[INFO] Converted 'verified_purchase' to integer type
[INFO] Applied log1p transformation to 'helpful_vote'
[INFO] Spam detection: Removed 7690 spam users
[INFO] Rows: 997285 -> 799211
[INFO] Spam detection: Removed 7690 spam users
[INFO] Rows: 997285 -> 799211
[INFO] Text filtering: 719978 valid texts for NLP processing
[INFO] Marked 79233 short texts as None (< 5 words)
[INFO] Truncated 26563 long texts to 250 words
[INFO] Text filtering: 719978 valid texts for NLP processing
[INFO] Marked 79233 short texts as None (<

In [4]:
train_df = add_user_segment(train_df)
train_df = add_features(train_df)
train_df, scaler = normalize(train_df, 'helpful_vote')

val_df = add_user_segment(val_df)
val_df = add_features(val_df)
val_df, _ = normalize(val_df, 'helpful_vote', scaler=scaler)

test_df = add_user_segment(test_df)
test_df = add_features(test_df)
test_df, _ = normalize(test_df, 'helpful_vote', scaler=scaler)

[INFO] User segments:
user_segment
Medium    38195
Heavy     15242
Light       471
Name: count, dtype: int64
[INFO] Added features: user_verified_ratio, item_avg_rating, is_weekend
[INFO] Normalized 'helpful_vote' (fitted) -> range [0, 1]
[INFO] User segments:
user_segment
Light     3091
Medium    2119
Heavy       50
Name: count, dtype: int64
[INFO] Added features: user_verified_ratio, item_avg_rating, is_weekend
[INFO] Normalized 'helpful_vote' (applied) -> range [0, 1]
[INFO] User segments:
user_segment
Light     2668
Medium    1950
Heavy       59
Name: count, dtype: int64
[INFO] Added features: user_verified_ratio, item_avg_rating, is_weekend
[INFO] Normalized 'helpful_vote' (applied) -> range [0, 1]


In [5]:
total_users = len(encoders['user_id'].classes_)
total_items = len(encoders['parent_asin'].classes_)

print(f"[INFO] Total Unique Users: {total_users}")
print(f"[INFO] Total Unique Items: {total_items}\n")

print("=== Train Matrix ===")
train_matrix = build_user_item_matrix(
    df=train_df, 
    total_users=total_users, 
    total_items=total_items,
    user_col='user_id',       
    item_col='parent_asin',   
    rating_col='rating'       
)

print("\n=== Validation Matrix ===")
val_matrix = build_user_item_matrix(
    df=val_df, 
    total_users=total_users, 
    total_items=total_items,
    user_col='user_id',
    item_col='parent_asin',
    rating_col='rating'
)

print("\n=== Test Matrix ===")
test_matrix = build_user_item_matrix(
    df=test_df, 
    total_users=total_users, 
    total_items=total_items,
    user_col='user_id',
    item_col='parent_asin',
    rating_col='rating'
)

[INFO] Total Unique Users: 7468
[INFO] Total Unique Items: 4218

=== Train Matrix ===
[INFO] Matrix built successfully with shape: (7468, 4218)
[INFO] Matrix sparsity: 99.8289%

=== Validation Matrix ===
[INFO] Matrix built successfully with shape: (7468, 4218)
[INFO] Matrix sparsity: 99.9833%

=== Test Matrix ===
[INFO] Matrix built successfully with shape: (7468, 4218)
[INFO] Matrix sparsity: 99.9852%


In [6]:
# Save processed data
processed_path = config['paths']['processed_data']

train_df.to_parquet(processed_path + 'train.parquet', index=False)
test_df.to_parquet(processed_path + 'val.parquet', index=False)
test_df.to_parquet(processed_path + 'test.parquet', index=False)

print(f"[INFO] Train saved: {train_df.shape}")
print(f"[INFO] Val saved:  {test_df.shape}")
print(f"[INFO] Test saved:  {test_df.shape}")

[INFO] Train saved: (53908, 14)
[INFO] Val saved:  (4677, 14)
[INFO] Test saved:  (4677, 14)


In [ ]:
import joblib

joblib.dump(encoders, processed_path + 'encoders.pkl')
joblib.dump(scaler, processed_path + 'scaler.pkl')

print("[INFO] Encoders and scaler saved.")

[INFO] Encoders and scaler saved ✅


In [8]:
import joblib

joblib.dump(train_matrix, processed_path + 'train_matrix.pkl')
joblib.dump(val_matrix, processed_path + 'val_matrix.pkl')
joblib.dump(test_matrix, processed_path + 'test_matrix.pkl')


['data/processed/test_matrix.pkl']